# InsPLAD Final Gold QA — Fixed Normal Reference

같은 고정 QA(`gold_v1`)에서 image-only baseline과 직접 비교합니다.

- Image 1: 같은 asset category의 검증된 정상 reference
- Image 2: target
- Q1/Q2/Q3 질문과 GT는 기존 baseline과 동일
- 기본 실행 모델: LLaVA-OneVision-7B / MiniCPM-V-4.5 / Qwen3.5-4B
- `RUN_ALL_FIVE=True`로 InternVL3.5-8B + Luxia까지 확장

결과는 `MyDrive/lvlm_insplad_results/gold_reference_v1/`에 저장합니다.


In [ ]:
# 0) 설정
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"

QA_FOLDER = "lvlm_insplad_qa/gold_v1"
BASELINE_FOLDER = "lvlm_insplad_results/gold_baseline_v1"
RESULT_FOLDER = "lvlm_insplad_results/gold_reference_v1"
DRIVE_DATA_DIR = None

RUN_ALL_FIVE = False
CORE_MODELS = [
    "llava_onevision_7b",
    "minicpm_v_4_5",
    "qwen3_5_4b",
]
ALL_MODELS = [
    "qwen3_5_4b",
    "internvl3_5_8b",
    "minicpm_v_4_5",
    "llava_onevision_7b",
    "luxia",
]
MODELS_TO_RUN = ALL_MODELS if RUN_ALL_FIVE else CORE_MODELS
CHECKPOINT_EVERY = 5

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
SUP_ZIP_NAME = "supervised_fault_classification.zip"
EXPECTED_ZIP_SIZES = {
    UNSUP_ZIP_NAME: 1_170_967_719,
    SUP_ZIP_NAME: 874_753_167,
}
print("models:", MODELS_TO_RUN)


In [ ]:
# 1) Drive mount + private repo clone + 의존성 설치
import base64, json, os, shutil, subprocess, sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
QA_DIR = DRIVE_ROOT / QA_FOLDER
BASELINE_DIR = DRIVE_ROOT / BASELINE_FOLDER
RESULT_DIR = DRIVE_ROOT / RESULT_FOLDER
RESULT_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:7]) + (" ..." if len(cmd) > 7 else ""))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 없습니다.")

auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.extraHeader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
    "GIT_TERMINAL_PROMPT": "0",
})
repo_url = f"https://github.com/{REPO}.git"
probe = subprocess.run(
    ["git", "ls-remote", "--heads", repo_url, REPO_REF],
    env=git_env, capture_output=True, text=True
)
if probe.returncode != 0:
    raise RuntimeError(f"GitHub 인증 실패: {(probe.stderr or '').strip()[:1200]}")
if not probe.stdout.strip():
    raise RuntimeError(f"branch '{REPO_REF}'를 찾지 못했습니다.")
run(["git", "clone", "--depth", "1", "--branch", REPO_REF, repo_url, REPO_DIR], env=git_env)
del auth, git_env, github_token

run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==5.14.0", "accelerate==1.14.0",
    "sentencepiece==0.2.2", "pillow", "pandas", "tqdm"
])
print("repo:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())
print("QA:", QA_DIR)
print("results:", RESULT_DIR)


In [ ]:
# 2) Secrets + GPU 확인
import torch

for secret_name in ["HF_TOKEN", "LUXIA_API_KEY"]:
    value = None
    try:
        value = userdata.get(secret_name)
    except Exception:
        pass
    if value:
        os.environ[secret_name] = value
        print("PASS secret:", secret_name)
    else:
        print("WARN missing secret:", secret_name)

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(props.total_memory / (1024**3), 1))
else:
    print("GPU 없음: HF 모델 실행 불가. Luxia만 API로 실행 가능.")


In [ ]:
# 3) QA + 두 source ZIP + reference 구조 확인
import pandas as pd

def find_unique_file(filename):
    roots = []
    if DRIVE_DATA_DIR:
        roots.append(DRIVE_ROOT / DRIVE_DATA_DIR)
    roots.append(DRIVE_ROOT)
    matches = []
    for root in roots:
        if not root.exists():
            continue
        direct = root / filename
        if direct.is_file():
            matches.append(direct)
        matches.extend(p for p in root.rglob(filename) if p.is_file())
    unique, seen = [], set()
    for p in matches:
        key = str(p.resolve())
        if key not in seen:
            unique.append(p); seen.add(key)
    if not unique:
        raise FileNotFoundError(f"{filename} 을 Drive에서 찾지 못했습니다.")
    if len(unique) > 1:
        print("[주의] 여러 후보 중 첫 파일 사용:", unique[0])
    return unique[0]

if not QA_DIR.is_dir():
    raise FileNotFoundError(f"QA folder 없음: {QA_DIR}")
UNSUP_ZIP = find_unique_file(UNSUP_ZIP_NAME)
SUP_ZIP = find_unique_file(SUP_ZIP_NAME)
for p in [UNSUP_ZIP, SUP_ZIP]:
    if p.stat().st_size != EXPECTED_ZIP_SIZES[p.name]:
        raise RuntimeError(f"ZIP size mismatch: {p}")
    print("PASS ZIP:", p)

sys.path.insert(0, str(REPO_DIR / "scripts"))
import run_gold_reference_baselines as ref_runner

rows = ref_runner.base.load_gold_qa(QA_DIR)
manifest = ref_runner.reference_manifest_rows(rows)
summary = ref_runner.reference_summary_rows(manifest)
q_counts = {}
for row in rows:
    q_counts[row["question_type"]] = q_counts.get(row["question_type"], 0) + 1
print("QA counts:", q_counts)
print("총 questions:", len(rows))
print("QA SHA256:", ref_runner.base.qa_identity(rows))
print("Reference SHA256:", ref_runner.reference_identity(manifest))
display(pd.DataFrame(summary))


In [ ]:
# 4) Fixed Normal Reference 실행 — 끊기면 이 셀만 다시 실행
cmd = [
    sys.executable,
    REPO_DIR / "scripts/run_gold_reference_baselines.py",
    "--qa-dir", QA_DIR,
    "--sup-zip", SUP_ZIP,
    "--unsup-zip", UNSUP_ZIP,
    "--out-dir", RESULT_DIR,
    "--checkpoint-every", str(CHECKPOINT_EVERY),
    "--models", *MODELS_TO_RUN,
]
run(cmd)


In [ ]:
# 5) Reference 결과 + Image-only baseline 대비 변화
import pandas as pd

ref_lb_path = RESULT_DIR / "leaderboard.csv"
base_lb_path = BASELINE_DIR / "leaderboard.csv"
status_path = RESULT_DIR / "run_status.json"

if status_path.is_file():
    print(json.dumps(json.loads(status_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))

ref_lb = pd.read_csv(ref_lb_path)
display(ref_lb.sort_values(["anomaly_f1", "defect_macro_f1"], ascending=False))

if base_lb_path.is_file():
    base_lb = pd.read_csv(base_lb_path)
    cols = ["model_key", "overall_accuracy", "anomaly_f1", "defect_macro_f1", "localization_accuracy"]
    cmp = base_lb[cols].merge(ref_lb[cols], on="model_key", suffixes=("_image", "_reference"))
    for metric in ["overall_accuracy", "anomaly_f1", "defect_macro_f1", "localization_accuracy"]:
        cmp[f"delta_{metric}"] = cmp[f"{metric}_reference"] - cmp[f"{metric}_image"]
    cmp.to_csv(RESULT_DIR / "baseline_vs_reference.csv", index=False)
    print()
    print("Image-only → Fixed Normal Reference 변화")
    display(cmp.sort_values("delta_anomaly_f1", ascending=False))
else:
    print("기존 baseline leaderboard가 없어 delta 비교는 생략합니다:", base_lb_path)


In [ ]:
# 6) Asset별 변화 확인 — 특히 glass-insulator / vari-grip
ref_asset_path = RESULT_DIR / "metrics_by_asset.csv"
base_asset_path = BASELINE_DIR / "metrics_by_asset.csv"
if ref_asset_path.is_file() and base_asset_path.is_file():
    ref_asset = pd.read_csv(ref_asset_path)
    base_asset = pd.read_csv(base_asset_path)
    cols = ["model_key", "asset_category", "accuracy", "q1_accuracy", "q2_accuracy"]
    asset_cmp = base_asset[cols].merge(ref_asset[cols], on=["model_key", "asset_category"], suffixes=("_image", "_reference"))
    for metric in ["accuracy", "q1_accuracy", "q2_accuracy"]:
        asset_cmp[f"delta_{metric}"] = asset_cmp[f"{metric}_reference"] - asset_cmp[f"{metric}_image"]
    asset_cmp.to_csv(RESULT_DIR / "asset_baseline_vs_reference.csv", index=False)
    display(asset_cmp.sort_values(["asset_category", "model_key"]))
else:
    print("asset 비교 파일이 아직 없습니다.")


In [ ]:
# 7) Prediction review CSV 생성
qa_rows = ref_runner.base.read_jsonl(RESULT_DIR / "run_qa.jsonl")
qa_by_id = {row["id"]: row for row in qa_rows}
review_rows = []
for model_key in MODELS_TO_RUN:
    pred_path = RESULT_DIR / "predictions" / f"{model_key}.jsonl"
    if not pred_path.is_file():
        continue
    for pred in ref_runner.base.read_jsonl(pred_path):
        qa = qa_by_id[pred["id"]]
        review_rows.append({
            "model_key": model_key,
            "id": qa["id"],
            "core_id": qa["core_id"],
            "asset_category": qa["asset_category"],
            "question_type": qa["question_type"],
            "gt_choice": qa["answer"],
            "predicted_choice": pred.get("predicted_choice"),
            "correct": pred.get("predicted_choice") == qa["answer"],
            "anomaly_gt": qa["anomaly_gt"],
            "defect_gt": qa["defect_gt"],
            "normal_reference_image": qa["normal_reference_image"],
            "target_image": qa["image"],
            "answer_text": qa.get("answer_text", ""),
            "raw_output": pred.get("output", ""),
            "error": pred.get("error"),
            "latency_s": pred.get("latency_s"),
        })
review = pd.DataFrame(review_rows)
review.to_csv(RESULT_DIR / "prediction_review.csv", index=False)
print("saved:", RESULT_DIR / "prediction_review.csv")
print("rows:", len(review))
if len(review):
    display(review[review["correct"] == False].head(50))


## 이 결과 다음 판단

- Reference가 Q1/Q2를 안정적으로 올리면 → 다음 `Vision Expert` 실험에서도 Reference 조합을 유지
- 특히 `glass-insulator`, `vari-grip`이 많이 오르는지 확인
- Reference 효과가 작거나 떨어지면 → Reference를 주 방법으로 쓰지 않고 Vision Expert로 바로 진행
- Q3는 현재 localization GT가 있는 subset만 해석하고, 전체 5개 asset localization 성능으로 표현하지 않음

기본 3모델 결과가 좋으면 `RUN_ALL_FIVE=True`로 바꿔 같은 notebook을 다시 실행하면 InternVL/Luxia가 이어서 추가됩니다.
